# Original Code (adapted for mac mps)

# TODO - currently script is for 2 stages

# find a way to get the output of the first pass only on the original image, overlay actual mask to see what is covered and what is cut out on the first pass and especially with images where the main nodule was not even captured
# rank by level of inaccurate, excluded pixels (TP / TP + FN)) testing recall in the first pass over 
# reference this article for top-k segmentation : https://pmc.ncbi.nlm.nih.gov/articles/PMC12428111/

In [8]:
# set constants for sizes of model output 
from tnscui_utils.TNSCUI_preprocess import TNSCUI_preprocess
import csv
import os
import traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import ttach as tta
from PIL import Image
from skimage.measure import label as sklabel
from skimage.measure import regionprops
from skimage.transform import resize
import segmentation_models_pytorch_4TorchLessThan120 as smp

# LOAD DATA CONFIGS 
PROJECT_ROOT = Path(
    "/Users/JanayeCheong/Documents/radiomics_segmentation_models/"
    "TNSCUI2020-Seg-Rank1st"
)

IMG_DIR = PROJECT_ROOT / "train_thyroidXL" / "raw_images"
MASK_DIR = PROJECT_ROOT / "train_thyroidXL" / "masks"

WEIGHT_C1 = (
    PROJECT_ROOT
    / "weigh_and_id"
    / "TNSCUI"
    / "fold1_stage1_trained_on_size_256.pkl"
)

OUTPUT_DIR = PROJECT_ROOT / "inference_outputs_pass_one"
PREDICTION_DIR = OUTPUT_DIR / "predicted_masks"
OVERLAY_DIR = OUTPUT_DIR / "overlays"
METRICS_CSV = OUTPUT_DIR / "stage1_metrics.csv"

C1_SIZE = 256

C1_TTA = True

ORIMG = False
C1_THRESHOLD = 0.5

SAVE_OVERLAYS = True
CONTINUE_ON_ERROR = True
MAX_IMAGES = None  # e.g. 10 for a quick test; None runs the full dataset

SUPPORTED_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}

In [9]:
def thyroidxl_preprocess(
    image_path: Path,
    outputsize: C1_SIZE, # C1_SIZE variable should be set to 256 x 256 
    # equivalent to TNSCUI_preprocess and removal of 1) black edges 2) irrelevant areas 
    # double check whether functions similarly ot the MATLAB process 
    remove_black_edges: bool = True,
    
    # function works by finding the first valid row pixels and column pixels, then retaining image on a foreground
):
    """
    ThyroidXL dataset safe replacement for TNSCUI_preprocess.

    Returns
    -------
    processed_tensor:
        Float tensor with shape [output_size, output_size].

    cut_shape:
        Shape of the retained image before resizing (to typical image tensor size).

    original_shape:
        Original image shape: (height, width).

    location:
        Coordinates of the retained image in the original image (the start and end of valid rows and columns):
        [row_start, row_end, col_start, col_end].
    """

    # Force the image into one grayscale channel.
    with Image.open(image_path) as image:
        image = image.convert("L") # convert grayscale channel?? 
        image_array = np.asarray(image, dtype=np.float32) # createa a numpy array of the image based on pixels and grayscale

    original_shape = image_array.shape

    if image_array.ndim != 2:
        raise ValueError(
            f"Expected a 2-D grayscale image, got {image_array.shape} "
            f"for {image_path.name}"
        ) # CHECK THAT THE IMAGE IS 2-DIMENSIONAL 

    if remove_black_edges: ## TODO: FIND BETTER WAY TO REMOVE BLACK EDGES
        # Detect rows and columns that contain meaningful ultrasound content.
        #
        # A small threshold is used instead of requiring pixels to be exactly
        # zero because ultrasound borders may contain compression noise.
        foreground_threshold = 5.0 # adjust for sensitivity to the saturation on the gray scale 

        valid_rows = np.where(
            np.mean(image_array, axis=1) > foreground_threshold # calculate image areas where the average of ROW pixels is greater than threshold 
        )[0]

        valid_cols = np.where(
            np.mean(image_array, axis=0) > foreground_threshold # calculate image areas where the average of COLUMN pixels is greater than threshold
        )[0]

        if len(valid_rows) > 0 and len(valid_cols) > 0: # find the foreground of the image based on the first valid row and the last + 1 valid row
            row_start = int(valid_rows[0])
            row_end = int(valid_rows[-1]) + 1

            col_start = int(valid_cols[0]) # do the same for the columns 
            col_end = int(valid_cols[-1]) + 1
        else:
            # Fall back to the complete image if no foreground is detected.
            row_start = 0
            row_end = original_shape[0]
            col_start = 0
            col_end = original_shape[1]

    else:
        row_start = 0
        row_end = original_shape[0] # x value of original shape (row end / max)
        col_start = 0
        col_end = original_shape[1] # y value of original shape (column end)

    cropped_image = image_array[
        row_start:row_end,
        col_start:col_end,
        
    ] # array is now cropped without black edges 

    if cropped_image.size == 0:
        raise ValueError(
            f"Black-edge removal produced an empty image for "
            f"{image_path.name}"
        )

    cut_shape = cropped_image.shape
    location = [row_start, row_end, col_start, col_end]

    # Resize to the stage-1 output_size (256 x 256) for the neural network.
    processed_array = resize(
        cropped_image,
        (outputsize, outputsize),
        order=3,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)

    # Match the common neural-network image range.
    if processed_array.max() > 1.0:
        processed_array /= 255.0

    processed_tensor = torch.from_numpy(processed_array) # convert the image to a tensor (different from TNSCUI_preprocess where originally converted to tensor)

    return processed_tensor, cut_shape, original_shape, location

In [10]:

# Metrics SAME AS original TNSCUI Model 
def get_iou(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate intersection over union for two binary arrays --> corresponds to ."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    union = np.logical_or(prediction, ground_truth).sum()

    if union == 0:
        return 1.0

    return float(intersection / union)


def get_dsc(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate Dice similarity coefficient for two binary arrays."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    denominator = prediction.sum() + ground_truth.sum()

    if denominator == 0:
        return 1.0

    return float((2.0 * intersection) / denominator)


def largest_connected_component(binary_mask: np.ndarray) -> np.ndarray:
    """Retain only the largest foreground connected component from binary threshold."""
    binary_mask = binary_mask.astype(bool)

    if binary_mask.sum() == 0:
        return binary_mask.astype(np.float32)

    labeled_img, num_components = sklabel(
        binary_mask,
        connectivity=1,
        background=0,
        return_num=True,
    )

    if num_components == 1:
        return binary_mask.astype(np.float32)

    component_sizes = [
        np.sum(labeled_img == component_label)
        for component_label in range(1, num_components + 1)
    ]

    largest_label = int(np.argmax(component_sizes)) + 1
    return (labeled_img == largest_label).astype(np.float32)


def calculate_stage2_roi(
    stage1_mask: np.ndarray,
    c1_size: int = 256,
) -> Tuple[int, int, int, int]:
    """
    Calculate the expanded square ROI used as input to Stage 2.

    Returns
    -------
    row_min, row_max, col_min, col_max
    """
    if stage1_mask.sum() == 0:
        min_row, min_col, max_row, max_col = 0, 0, c1_size, c1_size
    else:
        region = regionprops(stage1_mask.astype(np.uint8))[0]
        min_row, min_col, max_row, max_col = region.bbox

    row_center = (max_row + min_row) // 2
    col_center = (max_col + min_col) // 2
    max_length = max(max_row - min_row, max_col - min_col)

    large_roi_threshold = int((c1_size / 256) * 80)
    large_roi_margin = int((c1_size / 256) * 19)
    small_roi_margin = int((c1_size / 256) * 31)

    if max_length > large_roi_threshold:
        expansion = large_roi_margin + max_length // 2
    else:
        expansion = small_roi_margin + max_length // 2

    row_min = max(0, row_center - expansion)
    row_max = min(c1_size, row_center + expansion)
    col_min = max(0, col_center - expansion)
    col_max = min(c1_size, col_center + expansion)

    # IN case the crop is empty    
    if row_max <= row_min or col_max <= col_min:
        return 0, c1_size, 0, c1_size

    return row_min, row_max, col_min, col_max


# save files in order 

def discover_images(image_dir: Path) -> List[Path]:
    """Return every supported image file recursively, in stable order."""
    files = [
        path
        for path in image_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    return sorted(files, key=lambda path: str(path).lower())


def build_mask_index(mask_dir: Path) -> Dict[str, Path]:
    """
    Index masks by filename stem, case-insensitively.

    The image and mask may have different filename extensions, but their stems
    must match
    """
    index: Dict[str, Path] = {}

    for path in discover_images(mask_dir):
        key = path.stem.lower()

        if key in index:
            raise ValueError(
                f"Duplicate mask stem '{path.stem}' found:\n"
                f"  {index[key]}\n"
                f"  {path}"
            )

        index[key] = path

    return index


def load_binary_mask(mask_path: Path, expected_shape: Tuple[int, int]) -> np.ndarray:
    """Read a mask as grayscale and convert it to a binary NumPy array."""
    mask = Image.open(mask_path).convert("L")
    mask_array = np.asarray(mask, dtype=np.float32)

    if mask_array.shape != expected_shape:
        raise ValueError(
            f"Ground-truth mask shape {mask_array.shape} does not match "
            f"original image shape {expected_shape} for {mask_path.name}."
        )

    return (mask_array > 0).astype(np.float32)


def save_binary_mask(mask: np.ndarray, output_path: Path) -> None:
    """Save a binary mask as an 8-bit PNG."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    mask_uint8 = (mask.astype(bool).astype(np.uint8) * 255)
    Image.fromarray(mask_uint8, mode="L").save(output_path)


def mask_bbox(mask: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    """
    Return the axis-aligned bounding box of foreground pixels.

    Returns (row_min, col_min, row_max, col_max) in regionprops format,
    where row_max/col_max are exclusive upper bounds.
    """
    if mask.astype(bool).sum() == 0:
        return None

    region = regionprops(mask.astype(np.uint8))[0]
    return region.bbox


def draw_bbox_on_rgb(
    rgb: np.ndarray,
    bbox: Tuple[int, int, int, int],
    color: Tuple[float, float, float] = (0.0, 0.85, 1.0),
    thickness: int = 2,
) -> None:
    """Draw a rectangle on an RGB float image in [0, 1]."""
    row_min, col_min, row_max, col_max = bbox
    height, width = rgb.shape[:2]

    row_min = max(0, int(row_min))
    col_min = max(0, int(col_min))
    row_max = min(height, int(row_max))
    col_max = min(width, int(col_max))

    if row_max <= row_min or col_max <= col_min:
        return

    for offset in range(thickness):
        top = row_min + offset
        bottom = row_max - 1 - offset
        left = col_min + offset
        right = col_max - 1 - offset

        if top < height:
            rgb[top, col_min:col_max, :] = color
        if bottom >= 0:
            rgb[bottom, col_min:col_max, :] = color
        if left < width:
            rgb[row_min:row_max, left, :] = color
        if right >= 0:
            rgb[row_min:row_max, right, :] = color


def save_overlay(
    image_path: Path,
    prediction: np.ndarray,
    ground_truth: Optional[np.ndarray],
    output_path: Path,
    prediction_bbox: Optional[Tuple[int, int, int, int]] = None,
    bbox_color: Tuple[float, float, float] = (0.0, 0.85, 1.0),
    bbox_thickness: int = 2,
) -> None:
    """
    Save an RGB overlay:
    - red: the prediction
    - green: the ground truth (from given mask)
    - yellow: overlap
    - cyan box: optional bounding box showing where the prediction lands
    """
    original = Image.open(image_path).convert("L")
    base = np.asarray(original, dtype=np.float32)

    if base.max() > base.min():
        base = (base - base.min()) / (base.max() - base.min())
    else:
        base = np.zeros_like(base)

    rgb = np.stack([base, base, base], axis=-1)
    prediction_bool = prediction.astype(bool)

    rgb[prediction_bool, 0] = 1.0
    rgb[prediction_bool, 1] *= 0.35
    rgb[prediction_bool, 2] *= 0.35

    if ground_truth is not None:
        ground_truth_bool = ground_truth.astype(bool)
        rgb[ground_truth_bool, 1] = 1.0
        rgb[ground_truth_bool, 0] *= 0.35
        rgb[ground_truth_bool, 2] *= 0.35

        overlap = np.logical_and(prediction_bool, ground_truth_bool)
        rgb[overlap, 0] = 1.0
        rgb[overlap, 1] = 1.0
        rgb[overlap, 2] = 0.0

    if prediction_bbox is None:
        prediction_bbox = mask_bbox(prediction)

    if prediction_bbox is not None:
        draw_bbox_on_rgb(
            rgb,
            prediction_bbox,
            color=bbox_color,
            thickness=bbox_thickness,
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8), mode="RGB").save(
        output_path
    )


# ============================================================================
# Model loading and inference
# ============================================================================

def choose_device() -> torch.device:
    """Prefer Apple MPS, then CUDA, then CPU."""
    if torch.backends.mps.is_available():
        return torch.device("mps")

    if torch.cuda.is_available():
        return torch.device("cuda")

    return torch.device("cpu")


def extract_state_dict(checkpoint):
    """
    Support either a direct state_dict or a checkpoint dictionary containing
    a state_dict/model_state_dict field.
    """
    if not isinstance(checkpoint, dict):
        return checkpoint

    if "state_dict" in checkpoint:
        return checkpoint["state_dict"]

    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]

    return checkpoint


def load_model(
    weight_path: Path,
    device: torch.device,
    transforms,
    use_tta: bool,
) -> torch.nn.Module:
    """Create a DeepLabV3+ model and load pretrained weights."""
    if not weight_path.exists():
        raise FileNotFoundError(f"Weight file not found: {weight_path}")

    model = smp.DeepLabV3Plus(
        encoder_name="efficientnet-b6",
        encoder_weights=None,
        in_channels=1,
        classes=1,
    )

    # CPU deserialization is typically the safest for old checkpoints.
    checkpoint = torch.load(weight_path, map_location="cpu")
    state_dict = extract_state_dict(checkpoint)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as exc:
        print(
            f"Strict loading failed for {weight_path.name}; "
            "retrying with strict=False."
        )
        print(exc)
        incompatible = model.load_state_dict(state_dict, strict=False)
        print("Missing keys:", incompatible.missing_keys)
        print("Unexpected keys:", incompatible.unexpected_keys)

    model = model.to(device)
    model.eval()

    if use_tta:
        model = tta.SegmentationTTAWrapper(
            model,
            transforms,
            merge_mode="mean",
        )
        model.eval()

    return model

### TODO: IMPLEMENT THE OUTPUT FROM THE FIRST PASS OF THE MODEL (STAGE 1 MASK)

### FUNCTION BELOW OUTPUTS FROM THE FINAL MASK AND STAGE 1 MASK (256x256)


# def run_single_image(
#     image_path: Path,
#     model_cascade1: torch.nn.Module,
#     model_cascade2: torch.nn.Module,
#     device: torch.device,
# ) -> Tuple[np.ndarray, np.ndarray]:
#     """
#     Run both cascades on one image.

#     Returns
#     -------
#     final_mask:
#         Binary mask restored to the original image dimensions.
#     stage1_mask:
#         Binary 256x256 Stage-1 largest-component mask.
#     """
#     processed_img, cut_shape, original_shape, location = thyroidxl_preprocess(
#     image_path,
#     outputsize=C1_SIZE,
#     remove_black_edges=not ORIMG,
#     )
#     # Convert to a tensor if preprocessing returned a NumPy array.
#     if processed_img.ndim != 2:
#         raise ValueError(
#             f"Expected processed image shape [H, W], got "
#             f"{tuple(processed_img.shape)} for {image_path.name}"
#         )

#     image_tensor = processed_img.unsqueeze(0).unsqueeze(0)
#     image_tensor = image_tensor.to(
#         device=device,
#         dtype=torch.float32,
#     )

#     assert image_tensor.shape == (1, 1, C1_SIZE, C1_SIZE), (
#         f"Incorrect Stage-1 input shape: {tuple(image_tensor.shape)}"
#     )

#     image_array_256 = processed_img.cpu().numpy()

#     with torch.inference_mode():
#         stage1_logits = model_cascade1(image_tensor)
#         stage1_probability = torch.sigmoid(stage1_logits)

#     stage1_mask = (
#         stage1_probability.squeeze().detach().cpu().numpy() > C1_THRESHOLD
#     ).astype(np.float32)

#     stage1_mask = largest_connected_component(stage1_mask)

#     working_mask_256 = stage1_mask.copy()

#     if USE_C2:
#         row_min, row_max, col_min, col_max = calculate_stage2_roi(
#             stage1_mask,
#             C1_SIZE,
#         )

#         roi = image_array_256[row_min:row_max, col_min:col_max]
#         roi_original_shape = roi.shape

#         if roi.size == 0:
#             raise RuntimeError(
#                 f"Stage-2 ROI is empty for {image_path.name}: "
#                 f"{(row_min, row_max, col_min, col_max)}"
#             )

#         roi_512 = resize(
#             roi,
#             (C2_SIZE, C2_SIZE),
#             order=3,
#             preserve_range=True,
#             anti_aliasing=True,
#         ).astype(np.float32)

#         roi_tensor = torch.from_numpy(roi_512).unsqueeze(0).unsqueeze(0)
#         roi_tensor = roi_tensor.to(device=device, dtype=torch.float32)

#         with torch.inference_mode():
#             stage2_logits = model_cascade2(roi_tensor)
#             stage2_probability = torch.sigmoid(stage2_logits)

#         stage2_mask_512 = (
#             stage2_probability.squeeze().detach().cpu().numpy() > C2_THRESHOLD
#         ).astype(np.float32)

#         stage2_mask_roi = resize(
#             stage2_mask_512,
#             roi_original_shape,
#             order=C2_RESIZE_ORDER,
#             preserve_range=True,
#             anti_aliasing=False,
#         )

#         stage2_mask_roi = (stage2_mask_roi > 0.5).astype(np.float32)

#         # Use a blank canvas so the final 256x256 prediction contains only
#         # the refined Stage-2 output, rather than leftover Stage-1 pixels.
#         working_mask_256 = np.zeros((C1_SIZE, C1_SIZE), dtype=np.float32)
#         working_mask_256[row_min:row_max, col_min:col_max] = stage2_mask_roi

#     # Reverse the resize performed by TNSCUI_preprocess.
#     restored_cut_mask = resize(
#         working_mask_256,
#         tuple(int(value) for value in cut_shape),
#         order=0,
#         preserve_range=True,
#         anti_aliasing=False,
#     )

#     restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

#     original_shape = tuple(int(value) for value in original_shape)
#     final_mask = np.zeros(original_shape, dtype=np.float32)

#     row_start, row_end, col_start, col_end = [int(value) for value in location]
#     target_shape = (row_end - row_start, col_end - col_start)

#     if restored_cut_mask.shape != target_shape:
#         restored_cut_mask = resize(
#             restored_cut_mask,
#             target_shape,
#             order=0,
#             preserve_range=True,
#             anti_aliasing=False,
#         )
#         restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

#     final_mask[row_start:row_end, col_start:col_end] = restored_cut_mask
#     final_mask = (final_mask > 0.5).astype(np.float32)

#     return final_mask, stage1_mask
    
# """

In [11]:
# TODO : IMPLEMENT THE TNSCUI_preprocess4reesemble from the TNSCUI_prerpcoess file that remakes the image with mask in original dimensions from the map

def restore_mask_to_original(
    mask_256: np.ndarray,
    cut_shape: Tuple[int, int],
    original_shape: Tuple[int, int],
    location: List[int],
) -> np.ndarray:
    """Map a 256x256 binary mask back to original image coordinates."""
    restored_cut_mask = resize(
        mask_256,
        tuple(int(v) for v in cut_shape),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    )
    restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    original_shape = tuple(int(v) for v in original_shape)
    final_mask = np.zeros(original_shape, dtype=np.float32)

    row_start, row_end, col_start, col_end = [int(v) for v in location]
    target_shape = (row_end - row_start, col_end - col_start)

    if restored_cut_mask.shape != target_shape:
        restored_cut_mask = resize(
            restored_cut_mask,
            target_shape,
            order=0,
            preserve_range=True,
            anti_aliasing=False,
        )
        restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    final_mask[row_start:row_end, col_start:col_end] = restored_cut_mask
    return (final_mask > 0.5).astype(np.float32)


def get_recall(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Recall = TP / (TP + FN) — how much of the true nodule Stage 1 captured."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    true_positive = np.logical_and(prediction, ground_truth).sum()
    false_negative = np.logical_and(~prediction, ground_truth).sum()

    if true_positive + false_negative == 0:
        return 1.0

    return float(true_positive / (true_positive + false_negative))


def run_stage1_only(
    image_path: Path,
    model_cascade1: torch.nn.Module,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, Optional[Tuple[int, int, int, int]]]:
    """
    Run Stage 1 only (no Stage 2).

    Returns
    -------
    stage1_mask_original:
        Binary mask restored to original image dimensions.
    stage1_mask_256:
        Binary 256x256 Stage-1 largest-component mask.
    stage1_bbox_original:
        Bounding box of the restored Stage-1 mask in original image coordinates.
    """
    processed_img, cut_shape, original_shape, location = thyroidxl_preprocess(
        image_path,
        outputsize=C1_SIZE,
        remove_black_edges=not ORIMG,
    )

    image_tensor = processed_img.unsqueeze(0).unsqueeze(0).to(
        device=device,
        dtype=torch.float32,
    )

    assert image_tensor.shape == (1, 1, C1_SIZE, C1_SIZE), (
        f"Incorrect Stage-1 input shape: {tuple(image_tensor.shape)}"
    )

    with torch.inference_mode():
        stage1_logits = model_cascade1(image_tensor)
        stage1_probability = torch.sigmoid(stage1_logits)

    stage1_mask_256 = (
        stage1_probability.squeeze().detach().cpu().numpy() > C1_THRESHOLD
    ).astype(np.float32)
    stage1_mask_256 = largest_connected_component(stage1_mask_256)

    stage1_mask_original = restore_mask_to_original(
        stage1_mask_256,
        cut_shape,
        original_shape,
        location,
    )
    stage1_bbox_original = mask_bbox(stage1_mask_original)

    return stage1_mask_original, stage1_mask_256, stage1_bbox_original

In [12]:
# ============================================================================
# Main dataset loop — Stage 1 (first pass) only
# ============================================================================

def main() -> None:
    """Run Stage-1 inference only: 256 model, restore mask, overlay vs GT."""
    for required_path in (IMG_DIR, MASK_DIR, WEIGHT_C1):
        if not required_path.exists():
            raise FileNotFoundError(f"Path not found: {required_path}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

    if SAVE_OVERLAYS:
        OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

    device = choose_device()
    print(f"Using device: {device}")
    print(f"Image directory: {IMG_DIR}")
    print(f"Mask directory: {MASK_DIR}")

    image_files = discover_images(IMG_DIR)
    mask_index = build_mask_index(MASK_DIR)

    if not image_files:
        raise RuntimeError(f"No supported image files found in {IMG_DIR}")

    if MAX_IMAGES is not None:
        image_files = image_files[:MAX_IMAGES]
        print(f"Limiting run to first {len(image_files)} images (MAX_IMAGES={MAX_IMAGES}).")

    print(f"Processing {len(image_files)} images (Stage 1 only).")
    print(f"Found {len(mask_index)} masks.")

    missing_masks = [
        image_path.name
        for image_path in image_files
        if image_path.stem.lower() not in mask_index
    ]

    if missing_masks:
        preview = "\n".join(f"  - {name}" for name in missing_masks[:20])
        raise FileNotFoundError(
            f"{len(missing_masks)} images do not have matching masks by stem.\n"
            f"{preview}"
        )

    tta_transforms = tta.Compose(
        [
            tta.VerticalFlip(),
            tta.HorizontalFlip(),
            tta.Rotate90(angles=[0, 180]),
        ]
    )

    print("Loading Stage-1 model...")
    model_cascade1 = load_model(
        WEIGHT_C1,
        device,
        tta_transforms,
        C1_TTA,
    )

    results: List[Dict] = []
    failed_count = 0

    for index, image_path in enumerate(image_files, start=1):
        mask_path = mask_index[image_path.stem.lower()]

        print(
            f"\n[{index}/{len(image_files)}] {image_path.name} "
            f"(Stage 1 only, C1={C1_SIZE})"
        )

        try:
            original_image = Image.open(image_path)
            original_shape = (original_image.height, original_image.width)

            ground_truth = load_binary_mask(mask_path, original_shape)

            stage1_mask_original, stage1_mask_256, stage1_bbox = run_stage1_only(
                image_path,
                model_cascade1,
                device,
            )

            if stage1_mask_original.shape != ground_truth.shape:
                raise ValueError(
                    f"Prediction shape {stage1_mask_original.shape} does not match "
                    f"ground-truth shape {ground_truth.shape}."
                )

            iou = get_iou(stage1_mask_original, ground_truth)
            dsc = get_dsc(stage1_mask_original, ground_truth)
            recall = get_recall(stage1_mask_original, ground_truth)

            prediction_path = PREDICTION_DIR / f"{image_path.stem}_stage1.png"
            save_binary_mask(stage1_mask_original, prediction_path)

            if SAVE_OVERLAYS:
                overlay_path = OVERLAY_DIR / f"{image_path.stem}_stage1_overlay.png"
                save_overlay(
                    image_path,
                    stage1_mask_original,
                    ground_truth,
                    overlay_path,
                    prediction_bbox=stage1_bbox,
                )
            else:
                overlay_path = None

            bbox_text = ""
            if stage1_bbox is not None:
                row_min, col_min, row_max, col_max = stage1_bbox
                bbox_text = f"{row_min},{col_min},{row_max},{col_max}"

            result = {
                "image_name": image_path.name,
                "image_path": str(image_path),
                "mask_path": str(mask_path),
                "prediction_path": str(prediction_path),
                "overlay_path": str(overlay_path) if overlay_path else "",
                "height": original_shape[0],
                "width": original_shape[1],
                "stage1_bbox": bbox_text,
                "stage1_foreground_pixels_256": int(stage1_mask_256.sum()),
                "stage1_foreground_pixels_original": int(stage1_mask_original.sum()),
                "ground_truth_foreground_pixels": int(ground_truth.sum()),
                "iou": iou,
                "dsc": dsc,
                "recall": recall,
                "iou_below_0_3": int(iou < 0.3),
                "status": "ok",
                "error": "",
            }

            results.append(result)

            running_ious = [
                row["iou"] for row in results if row["status"] == "ok"
            ]
            running_dscs = [
                row["dsc"] for row in results if row["status"] == "ok"
            ]
            running_recalls = [
                row["recall"] for row in results if row["status"] == "ok"
            ]

            print(f"IoU: {iou:.4f}")
            print(f"DSC: {dsc:.4f}")
            print(f"Recall: {recall:.4f}")
            if stage1_bbox is not None:
                print(f"Stage-1 bbox (row_min,col_min,row_max,col_max): {stage1_bbox}")
            print(f"Running mean IoU: {np.mean(running_ious):.4f}")
            print(f"Running mean DSC: {np.mean(running_dscs):.4f}")
            print(f"Running mean recall: {np.mean(running_recalls):.4f}")

        except Exception as exc:
            failed_count += 1
            print(f"ERROR processing {image_path.name}: {exc}")
            traceback.print_exc()

            results.append(
                {
                    "image_name": image_path.name,
                    "image_path": str(image_path),
                    "mask_path": str(mask_path),
                    "prediction_path": "",
                    "overlay_path": "",
                    "height": "",
                    "width": "",
                    "stage1_bbox": "",
                    "stage1_foreground_pixels_256": "",
                    "stage1_foreground_pixels_original": "",
                    "ground_truth_foreground_pixels": "",
                    "iou": "",
                    "dsc": "",
                    "recall": "",
                    "iou_below_0_3": "",
                    "status": "failed",
                    "error": str(exc),
                }
            )

            if not CONTINUE_ON_ERROR:
                break

    fieldnames = [
        "image_name",
        "image_path",
        "mask_path",
        "prediction_path",
        "overlay_path",
        "height",
        "width",
        "stage1_bbox",
        "stage1_foreground_pixels_256",
        "stage1_foreground_pixels_original",
        "ground_truth_foreground_pixels",
        "iou",
        "dsc",
        "recall",
        "iou_below_0_3",
        "status",
        "error",
    ]

    with METRICS_CSV.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    successful_results = [
        row for row in results if row["status"] == "ok"
    ]

    print("\n" + "=" * 72)
    print("STAGE-1 ONLY INFERENCE COMPLETE")
    print("=" * 72)
    print(f"Images discovered: {len(image_files)}")
    print(f"Successfully processed: {len(successful_results)}")
    print(f"Failed: {failed_count}")

    if successful_results:
        ious = np.asarray(
            [row["iou"] for row in successful_results],
            dtype=np.float64,
        )
        dscs = np.asarray(
            [row["dsc"] for row in successful_results],
            dtype=np.float64,
        )
        recalls = np.asarray(
            [row["recall"] for row in successful_results],
            dtype=np.float64,
        )

        print(f"Mean IoU: {ious.mean():.4f}")
        print(f"Median IoU: {np.median(ious):.4f}")
        print(f"Mean DSC: {dscs.mean():.4f}")
        print(f"Median DSC: {np.median(dscs):.4f}")
        print(f"Mean recall: {recalls.mean():.4f}")
        print(f"Median recall: {np.median(recalls):.4f}")
        print(f"IoU below 0.3: {int(np.sum(ious < 0.3))}")
        print(f"Recall below 0.5: {int(np.sum(recalls < 0.5))}")

    print(f"Metrics CSV: {METRICS_CSV}")
    print(f"Predicted masks: {PREDICTION_DIR}")

    if SAVE_OVERLAYS:
        print(f"Overlays: {OVERLAY_DIR}")


main()


Using device: mps
Image directory: /Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/train_thyroidXL/raw_images
Mask directory: /Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/train_thyroidXL/masks
Processing 9541 images (Stage 1 only).
Found 9541 masks.
Loading Stage-1 model...

[1/9541] 00000058_2201CE11_0.png (Stage 1 only, C1=256)
IoU: 0.8440
DSC: 0.9154
Recall: 0.8442
Stage-1 bbox (row_min,col_min,row_max,col_max): (139, 261, 324, 503)
Running mean IoU: 0.8440
Running mean DSC: 0.9154
Running mean recall: 0.8442

[2/9541] 00000058_A73CED93_1.png (Stage 1 only, C1=256)
IoU: 0.7937
DSC: 0.8850
Recall: 0.7997
Stage-1 bbox (row_min,col_min,row_max,col_max): (112, 267, 305, 582)
Running mean IoU: 0.8188
Running mean DSC: 0.9002
Running mean recall: 0.8219

[3/9541] 00000127_1914A778_1.png (Stage 1 only, C1=256)
IoU: 0.7315
DSC: 0.8449
Recall: 0.7316
Stage-1 bbox (row_min,col_min,row_max,col_max): (69, 250, 160, 457)
Ru

KeyboardInterrupt: 

# Observations

1. mean recall for images = 0.73
    - large portion of TP are not accounted for 
    - poor coverage by first localization before expansion to margin (recall at this stage has not been observed)

2. 